In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv, find_dotenv

In [ ]:
load_dotenv(find_dotenv())


model = ChatGroq(model="qwen/qwen3.8-27b")

In [ ]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [ ]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic: {state["topic"]}'
    response = model.invoke(prompt)
    return {"joke": response.content}

def generate_explanation(state:JokeState):
    prompt = f'write an explanation for the joke:\n{state["joke"]}'
    response = model.invoke(prompt)
    return {"explanation": response.content}

In [ ]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable": {"thread_id": "user1"}}
workflow.invoke({"topic": "neutrinos"}, config=config1)

In [ ]:
list(workflow.get_state(config=config1))

In [ ]:
list(workflow.get_state_history(config=config1))

## Time Travel

In [ ]:
workflow.get_state({"configurable":{"thread_id":"user1", "checkpoint_id":"1f1d5706-59fe-6a36-8001-f9cb90490fca"}})

In [ ]:
workflow.invoke(None, {"configurable":{"thread_id":"user1", "checkpoint_id":"1f1d5706-59fe-6a36-8001-f9cb90490fca"}})
# starts from the 2nd node and generates the explaination again

In [ ]:
list(workflow.get_state_history(config1))

## Updating State

In [ ]:
workflow.update_state({"configurable":{"thread_id":"user1", 'checkpoint_ns': '', "checkpoint_id":"1f1d5706-5733-6cde-8000-af394074b9b7"}}, {"topic": "higgs bozon"})

In [ ]:
list(workflow.get_state_history(config1))

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "user1", "checkpoint_id": "1f1d572d-96a5-63a0-8001-ea27fa2d4b0a"}})

In [ ]:
list(workflow.get_state_history(config1))

## Fault Tolerance

In [ ]:
import time

class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    done: bool

In [ ]:
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging...")
    time.sleep(10)
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [ ]:
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.set_finish_point("step_3")

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
graph

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

In [ ]:
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))